# Customising a simulation

Three things decide what a simulation produces:

| | what | how to change it |
|---|---|---|
| `Electrodes` | where the electrodes sit, which device records | presets, your own list, YAML |
| `Subject` | the person | fields, `Subject.random(seed)` |
| `SimParams` | the model's constants, one block per step | `SimParams().replace(...)` |

Needs `pip install myogen[examples]` (matplotlib).

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np

import myogen as mg

motion = mg.example_motion(duration=3.0, fs=100)

def rms(recording):
    return np.sqrt((recording.emg ** 2).mean(0))

## Electrodes

Each electrode is bipolar and placed anatomically: `angle_deg` around the forearm (0 radial, +90 dorsal over the extensors, -90 palmar over the flexors), `position_cm` from the elbow, `ied_mm` between the poles and `orientation_deg` of the pole axis (0 = along the fibres). The `raw` device gives mV at `fs`.

In [ ]:
electrodes = mg.Electrodes([
    mg.Electrode(angle_deg=90, position_cm=7, name="extensors"),
    mg.Electrode(angle_deg=-90, position_cm=7, name="flexors"),
    mg.Electrode(angle_deg=90, position_cm=7, orientation_deg=90, name="extensors, across"),
], device="raw")
recording = mg.Simulator(electrodes).run(motion)
for name, value in zip(electrodes.names, rms(recording)):
    print(f"{name:20s} {value * 1000:6.1f} uV RMS")

Electrode sets are device profiles: save them as YAML, edit them, and load them back (or pass the file to `myogen simulate --electrodes`).

In [ ]:
electrodes.save("my_electrodes.yml")
print(open("my_electrodes.yml").read()[:400])

## Subjects

`Subject()` is an average adult. `Subject.random(seed)` draws one from population ranges (forearm size, fat, muscle sizes, effort, ...); `amputee=True` gives a transradial residual limb without intrinsic hand muscles.

In [ ]:
subjects = {"average": mg.Subject(), "random 3": mg.Subject.random(3),
            "amputee": mg.Subject.random(3, amputee=True)}
ring = mg.Electrodes.ring(n=8, position_cm=5, device="raw")
fig, ax = plt.subplots(figsize=(8, 3))
for name, subject in subjects.items():
    ax.plot(rms(mg.Simulator(ring, subject=subject).run(motion)) * 1000, "o-", label=name)
ax.set_xlabel("channel (around the forearm)")
ax.set_ylabel("uV RMS")
ax.legend()

## Model parameters

Every constant of the simulation is in `myogen.params`, one frozen dataclass per step. `replace` returns a copy with some fields changed. Here: a larger EMG decay distance makes every electrode see muscles from further away, so neighbouring channels become more alike.

In [ ]:
params = mg.SimParams()
print(params.emg)

wide = params.replace(emg={"decay_mm": 20.0})
for label, p in [("decay 12 mm", params), ("decay 20 mm", wide)]:
    emg = mg.Simulator(ring, params=p).run(motion).emg
    neighbours = np.mean([np.corrcoef(np.abs(emg[:, c]), np.abs(emg[:, (c + 1) % 8]))[0, 1] for c in range(8)])
    print(f"{label}: neighbour envelope correlation {neighbours:.2f}")

## Matching a real device

`calibrate` fits the device settings of an electrode profile (gain, noise, Otto Bock smoothing, optionally the armband rotation) to real recordings of known movements. Noise can only be measured where the muscles rest, so the recording needs some labelled rest (`HandMotion.rest`). Here a simulated recording with a different gain and noise stands in for the real one.

In [ ]:
from myogen.inputs.motion import REST_POSTURE
from myogen.simulation.musculoskeletal import HAND_JOINTS

relaxed = np.tile([REST_POSTURE[j] for j in HAND_JOINTS], (200, 1))       # 2 s of a relaxed hand
session = mg.HandMotion(np.vstack([relaxed, motion.angles]), motion.fs,
                        rest=np.r_[np.ones(200, bool), np.zeros(len(motion.angles), bool)])

sim = mg.Simulator(mg.Electrodes.otto_bock())
truth = dataclasses.replace(mg.Electrodes.otto_bock(), gain=1.7, noise_uv=3.0)
real = sim.record(sim.simulate_clean(session), truth).emg

cal = mg.calibrate(sim, [session], [real])
print(cal)
print("fitted gain:", round(cal.electrodes.gain, 2), " noise_uv:", round(cal.electrodes.noise_uv, 2))